In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [2]:
data_path = 'data/'
df_lists = []

customers_df = pd.read_csv(data_path + 'olist_customers_dataset.csv')
geolocation_df = pd.read_csv(data_path + 'olist_geolocation_dataset.csv')
order_items_df = pd.read_csv(data_path + 'olist_order_items_dataset.csv')
order_payments_df = pd.read_csv(data_path + 'olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(data_path + 'olist_order_reviews_dataset.csv')
orders_df = pd.read_csv(data_path + 'olist_orders_dataset.csv')
products_df = pd.read_csv(data_path + 'olist_products_dataset.csv')
sellers_df = pd.read_csv(data_path + 'olist_sellers_dataset.csv')
category_translation_df = pd.read_csv(data_path + 'product_category_name_translation.csv')

df_lists.append(customers_df)
df_lists.append(geolocation_df)
df_lists.append(order_items_df)
df_lists.append(order_payments_df)
df_lists.append(order_reviews_df)
df_lists.append(orders_df)
df_lists.append(products_df)
df_lists.append(sellers_df)
df_lists.append(category_translation_df)

df_names = ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']

### **Manejo de valores Nulos**

#### order_reviews_df

In [6]:
print("=== Análisis de nulos en order_reviews_df ===")
print("Valores nulos:")
print(order_reviews_df.isnull().sum())
print("\nPorcentaje de nulos:")
print((order_reviews_df.isnull().sum() / len(order_reviews_df)) * 100)

=== Análisis de nulos en order_reviews_df ===
Valores nulos:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
has_comment                    0
dtype: int64

Porcentaje de nulos:
review_id                   0.000000
order_id                    0.000000
review_score                0.000000
review_comment_title       88.341530
review_comment_message     58.702532
review_creation_date        0.000000
review_answer_timestamp     0.000000
has_comment                 0.000000
dtype: float64


Los nulos en las columnas `review_comment_title` (**88.34%**) y `review_comment_message` (**58.70%**) son esperados, ya que los comentarios son opcionales. Imputar con texto genérico podría sesgar análisis de texto, y eliminar filas eliminaría la mayoría de los datos.

Se creará una columna binaria `has_comment` para indicar si hay comentario en `review_comment_message`

In [5]:
order_reviews_df['has_comment'] = order_reviews_df['review_comment_message'].notnull().astype(int)

#### orders_df

In [7]:
print("\n=== Análisis de nulos en orders_df ===")
print("Valores nulos:")
print(orders_df.isnull().sum())
print("\nPorcentaje de nulos:")
print((orders_df.isnull().sum() / len(orders_df)) * 100)
print("\nDistribución de order_status para order_delivered_customer_date nulo:")
print(orders_df[orders_df['order_delivered_customer_date'].isnull()]['order_status'].value_counts())


=== Análisis de nulos en orders_df ===
Valores nulos:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Porcentaje de nulos:
order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

Distribución de order_status para order_delivered_customer_date nulo:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count

Los nulos en `order_approved_at`(**0.16%**), `order_delivered_carrier_date` (**1.79%**), y `order_delivered_customer_date` (**2.98%**) están relacionados con el estado del pedido. Según los datos de `order_status` para `order_delivered_customer_date` nulo, la mayoría corresponde a pedidos no entregados (**shipped**, **canceled**, **unavailable**, etc.), con solo 8 casos en delivered. Los nulos en `order_approved_at` y `order_delivered_carrier_date` también reflejan pedidos en etapas incompletas.

Se creará una columna binaria `delivered` para indicar si el pedido fue entregado

In [8]:
orders_df['delivered'] = orders_df['order_delivered_customer_date'].notnull().astype(int)

#### products_df

In [10]:
print("\n=== Análisis de nulos en products_df ===")
print("Valores nulos:")
print(products_df.isnull().sum())
print("\nPorcentaje de nulos:")
print((products_df.isnull().sum() / len(products_df)) * 100)


=== Análisis de nulos en products_df ===
Valores nulos:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Porcentaje de nulos:
product_id                    0.000000
product_category_name         1.851234
product_name_lenght           1.851234
product_description_lenght    1.851234
product_photos_qty            1.851234
product_weight_g              0.006070
product_length_cm             0.006070
product_height_cm             0.006070
product_width_cm              0.006070
dtype: float64


Los **610** nulos (**1.85%**) en `product_category_name`, `product_name_lenght`, `product_description_lenght`, y `product_photos_qty` corresponden a las mismas filas. Mientras que los nulos en dimensiones es únicamente **1**. 
Se imputarán:
- **Desconocido** para ``categorías``
- **0** para ``fotos``
- **medianas** para ``numéricos``

In [12]:
products_df['product_category_name']= products_df['product_category_name'].fillna('Desconocido')
products_df['product_photos_qty'] = products_df['product_photos_qty'].fillna(0)
for col in ['product_name_lenght', 'product_description_lenght', 'product_weight_g', 
            'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products_df[col] = products_df[col].fillna(products_df[col].median())

In [13]:
print("\n=== Resumen final de nulos ===")
for df, name in zip(df_lists, df_names):
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(f"\n{name}:")
        print(null_counts[null_counts > 0])
    else:
        print(f"\n{name}: No hay nulos.")


=== Resumen final de nulos ===

customers: No hay nulos.

geolocation: No hay nulos.

order_items: No hay nulos.

order_payments: No hay nulos.

order_reviews:
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products: No hay nulos.

sellers: No hay nulos.

category_translation: No hay nulos.


Los valores nulos que quedan ya están respaldados con las nuevas columnas agregadas.